# Block 1 on Kaggle: environment, repo, schema check

**Notebook settings (right panel) before running:**
- **Accelerator:** None for sections 1–4 (saves GPU quota). Switch to **GPU T4 x2** only for section 5. Do not pick P100: current vLLM needs compute capability 7.0+.
- **Internet:** On (requires a phone-verified Kaggle account).
- **Environment:** "Pin to original environment" once things work, so the image does not change under you.
- **Secrets (Add-ons → Secrets):** `GITHUB_TOKEN` (fine-grained, Contents read/write on your repo only) and `HF_TOKEN` (read). Attach both to this notebook.

Each group member runs their **own copy** of this notebook with their **own** secrets. Set `REPO` to the group repo and fill in your name.

In [ ]:
REPO = "YOUR_ORG/banglish-sanitizer"   # <-- the group's repo (same for everyone)
GIT_NAME = "Your Name"                # <-- each member edits their own copy
GIT_EMAIL = "you@example.com"
MEMBER = "yourname"                   # short id, used in branch and file names
WORK = "/kaggle/working/banglish-sanitizer"

import os, glob, shutil, subprocess, sys
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GH = secrets.get_secret("GITHUB_TOKEN")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
REMOTE = f"https://{GH}@github.com/{REPO}.git"

def sh(cmd, cwd=WORK):
    # runs a command without echoing the token-bearing remote URL
    r = subprocess.run(cmd, cwd=cwd, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(GH, "***"))
    if r.returncode: raise RuntimeError(f"failed: {cmd.replace(GH, '***')}")
print("Python", sys.version.split()[0])

## 1. One-time bootstrap (ONE group member runs this once; everyone else skips it)

Upload `banglish-sanitizer.zip` as a **private** Kaggle Dataset and add it as input to this notebook. Create an **empty** private GitHub repo (no README). This cell copies the skeleton, restores `.gitignore` if the upload dropped it, makes the first commit, tags the frozen schema and pushes.

Review `configs/schema.json` on your laptop **before** uploading: the tag below freezes it.

In [ ]:
RUN_BOOTSTRAP = False   # set True for the single first run

if RUN_BOOTSTRAP:
    hits = glob.glob("/kaggle/input/**/configs/schema.json", recursive=True)
    assert hits, "Skeleton not found: add the uploaded dataset as notebook input"
    src_root = os.path.dirname(os.path.dirname(hits[0]))
    shutil.copytree(src_root, WORK, dirs_exist_ok=True)
    gi = os.path.join(WORK, ".gitignore")
    if not os.path.exists(gi):
        open(gi, "w").write("data/raw/*\ndata/silver/*\ndata/gold/*\n!data/*/.gitkeep\n"
                            ".venv/\n__pycache__/\n*.pyc\n.pytest_cache/\ncheckpoints/\n"
                            "runs/\nwandb/\n*.safetensors\n*.bin\n.env\n")
    for d in ["data/raw", "data/silver", "data/gold"]:
        os.makedirs(os.path.join(WORK, d), exist_ok=True)
        open(os.path.join(WORK, d, ".gitkeep"), "a").close()
    sh("git init -b main")
    sh(f'git config user.name "{GIT_NAME}" && git config user.email "{GIT_EMAIL}"')
    sh("git add . && git commit -m 'Project skeleton and output schema v1.0.0'")
    sh("git tag schema-v1.0.0")
    sh(f"git push {REMOTE} main --tags")

## 2. Every session: get the latest code

In [ ]:
if not os.path.exists(os.path.join(WORK, ".git")):
    sh(f"git clone {REMOTE} {WORK}", cwd="/kaggle/working")
else:
    sh(f"git pull {REMOTE} main")
os.chdir(WORK)
sh(f'git config user.name "{GIT_NAME}" && git config user.email "{GIT_EMAIL}"')
sh("git log --oneline -3 && git tag")

## 3. Install the light dependencies and run the schema tests (CPU is enough)

In [ ]:
!pip install -q jsonschema datasketch pytest
!python -m pytest -q tests/

## 4. Record the Kaggle environment (reproducibility)

In [ ]:
import datetime
os.makedirs("env", exist_ok=True)
stamp = datetime.date.today().isoformat()
!pip freeze > env/kaggle_env_{MEMBER}_{stamp}.txt
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
for pkg in ["datasets", "peft", "accelerate"]:
    try: print(pkg, __import__(pkg).__version__)
    except ImportError: print(pkg, "NOT INSTALLED -> pip install", pkg)

Tokenizer / vocabulary check (Day 1, Block 3). Only downloads config and tokenizer files, so CPU is fine.

In [ ]:
from transformers import AutoTokenizer, AutoConfig
sample = "kalk3 bkash e 5oo tk send korsi trans id 9X87K"
for m in ["Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct"]:
    tok, cfg = AutoTokenizer.from_pretrained(m), AutoConfig.from_pretrained(m)
    print(f"{m}: tokenizer={len(tok)}  embedding_rows={cfg.vocab_size}")
    print("  ", tok.tokenize(sample))

## 5. vLLM + schema smoke test (switch Accelerator to GPU T4 x2 first)

Checks that the grammar engine accepts `configs/schema.json` and that a raw 1.5B model can already emit schema-valid JSON. The content will be poor; the point is that the pipeline works end to end.

Installing vLLM may replace Kaggle's torch. If imports fail after the install, restart the session, rerun sections 0 and 2, and skip the pip line.

In [ ]:
!pip install -q vllm

In [ ]:
import json
from vllm import LLM, SamplingParams
sys.path.insert(0, WORK)
from src.common.validate import validate

schema = json.load(open("configs/schema.json"))
try:   # newer vLLM API
    from vllm.sampling_params import StructuredOutputsParams
    sp = SamplingParams(temperature=0, max_tokens=600,
                        structured_outputs=StructuredOutputsParams(json=schema))
except ImportError:   # older vLLM API
    from vllm.sampling_params import GuidedDecodingParams
    sp = SamplingParams(temperature=0, max_tokens=600,
                        guided_decoding=GuidedDecodingParams(json=schema))

llm = LLM("Qwen/Qwen2.5-1.5B-Instruct", dtype="half", max_model_len=2048,
          gpu_memory_utilization=0.85)   # T4 has no bf16, so fp16 ("half")

src = ("kalk3 bkash e 5oo tk send korsi trans id 9X87K, taka pay nai. "
       "1987654321 eta check koren")
msg = [{"role": "system", "content": "Rewrite the user's Banglish prompt into clean English, "
        "mask PII with <TYPE_n> placeholders, list uncertain spans. Output JSON only."},
       {"role": "user", "content": src}]
text = llm.chat(msg, sp)[0].outputs[0].text
out = json.loads(text)
print(json.dumps(out, indent=2, ensure_ascii=False))
print("\nvalidator errors:", validate(out, src) or "none")

## 6. Save your work

`/kaggle/working` is wiped when the session ends. Commit to a personal branch and open a pull request on GitHub; nobody pushes to `main` directly. For large outputs later (silver data, checkpoints), use **Save Version** or a private Kaggle Dataset / Hugging Face repo, never git.

In [ ]:
BRANCH = f"{MEMBER}/env-{stamp}"
sh(f"git checkout -B {BRANCH}")
sh("git add env/ && git commit -m 'Record Kaggle environment' || true")
sh(f"git push {REMOTE} {BRANCH}")
print(f"Now open a pull request for branch {BRANCH} on GitHub.")